# 04 — PPO evaluation

Сравнение PPO со всеми baselines. `USE_TEST_SEEDS=True` разрешается только один раз после окончательного выбора модели.

In [ ]:
from pathlib import Path
import json
import sys
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path('/content/drive/MyDrive/gas_portfolio_rl')
except ImportError:
    IN_COLAB = False
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = cwd if (cwd / 'pyproject.toml').exists() else cwd / 'gas_portfolio_rl'
sys.path.insert(0, str(PROJECT_ROOT))
from notebook_bootstrap import bootstrap_project
PROJECT_ROOT = bootstrap_project(PROJECT_ROOT, install_dependencies=IN_COLAB, install_rl_dependencies=True)

In [ ]:
from gas_portfolio_rl import AppConfig
from gas_portfolio_rl.evaluation.ppo import evaluate_ppo_bundle
from gas_portfolio_rl.evaluation.acceptance import assess_ppo
from gas_portfolio_rl.evaluation import build_ppo_leaderboard, register_ppo_evaluation
from gas_portfolio_rl.evaluation.reporting import save_evaluation
from gas_portfolio_rl.evaluation.scenarios import make_scenarios
from gas_portfolio_rl.visualization import plot_evaluation_diagnostics, plot_training_diagnostics, plot_versioned_pnl_comparison
from IPython.display import display

QUICK_EVALUATION = True
EXPERIMENT_TAG = 'v5_full'  # Например: v4_reference, v5_control, v5_stable, v5_full.
USE_TEST_SEEDS = False  # Do not change while tuning PPO.
runs = sorted((PROJECT_ROOT / 'results/ppo' / EXPERIMENT_TAG).glob('run_*'))
if not runs:
    raise FileNotFoundError('No PPO runs. Execute 03_train_ppo.ipynb first.')
RUN_DIR = runs[-1]  # Replace explicitly when comparing several training seeds.
metadata = json.loads((RUN_DIR / 'run_metadata.json').read_text(encoding='utf-8'))
config = AppConfig.from_mapping(metadata['config'])  # Exact training reward, market and PPO settings.
REGISTRY_PATH = PROJECT_ROOT / 'results/ppo/registry/ppo_history.csv'
print('Evaluating:', RUN_DIR)
display(plot_training_diagnostics(RUN_DIR))

In [ ]:
count = 3 if QUICK_EVALUATION else config.evaluation.seeds
stress_count = 2 if QUICK_EVALUATION else config.evaluation.stress_seeds
seed_start = 20000 if USE_TEST_SEEDS else config.evaluation.seed_start
stress_seed_start = 40000 if USE_TEST_SEEDS else 30000
scenario_seeds = {'base': range(seed_start, seed_start + count), 'calm': range(seed_start, seed_start + count), 'crisis_stress': range(stress_seed_start, stress_seed_start + stress_count)}
result = evaluate_ppo_bundle(config, make_scenarios(config), model_path=RUN_DIR / 'best_model.zip', normalization_path=RUN_DIR / 'best_vecnormalize.pkl', seeds=scenario_seeds)
output = RUN_DIR / ('test_evaluation' if USE_TEST_SEEDS else 'validation_evaluation')
save_evaluation(result, output)
acceptance = assess_ppo(result, config)
with (output / 'acceptance.json').open('w', encoding='utf-8') as stream:
    json.dump(acceptance, stream, indent=2)
evaluation_split = 'test' if USE_TEST_SEEDS else 'validation'
registry = register_ppo_evaluation(result, run_dir=RUN_DIR, config=config, acceptance=acceptance, registry_path=REGISTRY_PATH, evaluation_split=evaluation_split)
diagnostics = plot_evaluation_diagnostics(result.summary, initial_value=config.portfolio.initial_cash)
diagnostics.savefig(output / 'evaluation_diagnostics.png', dpi=150)
display(diagnostics)
versioned_pnl = plot_versioned_pnl_comparison(result.summary, registry, initial_value=config.portfolio.initial_cash, evaluation_split=evaluation_split)
versioned_pnl.savefig(output / 'versioned_net_pnl.png', dpi=150)
display(versioned_pnl)
leaderboard = build_ppo_leaderboard(registry, evaluation_split=evaluation_split)
print('PPO robustness leaderboard:')
for row in leaderboard:
    print(f"#{row['rank']} {row['experiment_tag']:18s} {row['run_id']} pass={row['passes_all']} worst_reward={row['worst_scenario_reward']:.3f} crisis_DD={row['crisis_max_drawdown']:.2%} worst_return={row['worst_scenario_return']:.2%}")
print('Acceptance:', acceptance)
for row in result.summary:
    print(f"{row['scenario']:14s} {row['strategy']:22s} return={row['total_return_mean']:8.2%} DD={row['max_drawdown_mean']:8.2%} reward={row['total_reward_mean']:8.3f} MC={row['margin_call_rate']:6.1%}")